# Replicação do Estudo H-CAT para Dataset de Diabetes

Este notebook replica os experimentos do paper "Dissecting Sample Hardness: A Fine-Grained Analysis of Hardness Characterization Methods for Data-Centric AI" para o dataset de diabetes.

## Objetivo
Aplicar o H-CAT no dataset tabular de diabetes e obter os mesmos resultados do paper (página 62).


In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import openml
from pathlib import Path

def _find_hcat_root() -> Path:
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        cand = base / "H-CAT"
        if (cand / "src").is_dir():
            return cand.resolve()
    raise FileNotFoundError(
        "HCAT nao encontrado"
    )

hcat_path = _find_hcat_root()
hcat_str = str(hcat_path)
if hcat_str not in sys.path:
    sys.path.insert(0, hcat_str)

from src.trainer import PyTorchTrainer
from src.dataloader import MultiFormatDataLoader
from src.models import MLP
from src.evaluator import Evaluator
from src.utils import seed_everything

In [ ]:
TOTAL_RUNS = 3
EPOCHS = 10
SEED = 0
SAMPLE_SIZE = 50000

HARDNESS_TYPES = ["uniform", "asymmetric", "adjacent", "instance", "atypical"]
PROPORTIONS = [0.1, 0.3, 0.5]

# Rule matrix e feature usadas por 'instance' e 'atypical'
RULE_MATRIX_DIABETES = {2: [0], 1: [0], 0: [1]}
ATYPICAL_FEATURE = "num_medications"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
dataset = openml.datasets.get_dataset(4541)
X, y, _, attribute_names = dataset.get_data(
    dataset_format="array",
    target=dataset.default_target_attribute,
)
df = pd.DataFrame(X, columns=attribute_names)
df["y"] = y
if len(df) > SAMPLE_SIZE:
    df = df.sample(SAMPLE_SIZE, random_state=SEED).reset_index(drop=True)


In [ ]:
def run_experiment(hardness_type, proportion, run_number, seed, df, device):
    seed_everything(seed)
    
    X = df.drop(columns="y").to_numpy()
    y = df["y"].values
    total_samples = len(df)
    num_classes = len(np.unique(y))
    
    if hardness_type == "instance":
        rule_matrix = RULE_MATRIX_DIABETES
    else:
        rule_matrix = None
    
    if hardness_type == "atypical":
        marginal = df[ATYPICAL_FEATURE].values
        index = df.columns.get_loc(ATYPICAL_FEATURE)
        atypical_marginal = (marginal, index)
    else:
        atypical_marginal = None
    
    data = (X, y)
    
    dataloader_class = MultiFormatDataLoader(
        data=data,
        target_column=None,
        data_type="numpy",
        data_modality="tabular",
        batch_size=64,
        shuffle=True,
        num_workers=0,
        transform=None,
        image_transform=None,
        perturbation_method=hardness_type,
        p=proportion,
        rule_matrix=rule_matrix,
        atypical_marginal=atypical_marginal,
    )
    
    dataloader, dataloader_unshuffled = dataloader_class.get_dataloader()
    flag_ids = dataloader_class.get_flag_ids()
    
    characterization_methods = [
        "aum",
        "data_uncert",
        "el2n",
        "grand",
        "cleanlab",
        "forgetting",
        "vog",
        "prototypicality",
        "loss",
        "conf_agree",
    ]
    
    model = MLP(input_size=X.shape[1], nlabels=num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    trainer = PyTorchTrainer(
        model=model,
        criterion=criterion,
        optimizer=optimizer,
        lr=0.001,
        epochs=EPOCHS,
        total_samples=total_samples,
        num_classes=num_classes,
        device=device,
        characterization_methods=characterization_methods,
    )
    
    trainer.fit(dataloader, dataloader_unshuffled)
    hardness_dict = trainer.get_hardness_methods()
    
    eval_dict = {}
    raw_scores_dict = {}
    
    for method in hardness_dict.keys():
        try:
            temp_eval = Evaluator(
                hardness_dict={method: hardness_dict[method]}, 
                flag_ids=flag_ids, 
                p=proportion
            )
            method_results, method_raw = temp_eval.compute_results()
            eval_dict.update(method_results)
            raw_scores_dict.update(method_raw)
        except Exception:
            continue
    
    results = {
        "hardness_type": hardness_type,
        "proportion": proportion,
        "run": run_number,
        "seed": seed,
        "metrics": eval_dict,
        "raw_scores": raw_scores_dict,
        "flag_ids": flag_ids,
    }
    
    return results

In [ ]:
all_results = []

for hardness_type in HARDNESS_TYPES:
    for prop in PROPORTIONS:
        for run in range(1, TOTAL_RUNS + 1):
            current_seed = SEED + (run - 1)
            
            try:
                result = run_experiment(
                    hardness_type=hardness_type,
                    proportion=prop,
                    run_number=run,
                    seed=current_seed,
                    df=df,
                    device=device
                )
                all_results.append(result)
            except Exception as e:
                import traceback
                traceback.print_exc()
                continue

## 6. Análise e Visualização dos Resultados

Analisando os resultados e comparando com os resultados do paper.


In [ ]:
results_list = []
for result in all_results:
    for method, metrics in result["metrics"].items():
        results_list.append({
            "hardness_type": result["hardness_type"],
            "proportion": result["proportion"],
            "run": result["run"],
            "method": method,
            "auc_roc": metrics.get("auc_roc", np.nan),
            "auc_prc": metrics.get("auc_prc", np.nan),
        })

results_df = pd.DataFrame(results_list)

In [ ]:
OUTPUT_DIR = Path("..") / "data" / "comparison_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = OUTPUT_DIR / "nb01_hcat_diabetes_results.csv"

results_to_save = results_df.rename(columns={"method": "measure"}).copy()
results_to_save.insert(0, "dataset", "diabetes")
results_to_save.insert(1, "method_family", "hcat")
results_to_save = results_to_save[[
    "dataset", "method_family", "hardness_type", "proportion", "run", "measure",
    "auc_roc", "auc_prc",
]]
results_to_save.to_csv(OUTPUT_CSV, index=False)


In [ ]:
summary = results_df.groupby(["hardness_type", "proportion", "method"]).agg({
    "auc_roc": ["mean", "std"],
    "auc_prc": ["mean", "std"]
}).round(4)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (15, 10)

for hardness_type in HARDNESS_TYPES:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    hardness_data = results_df[results_df["hardness_type"] == hardness_type]
    
    ax1 = axes[0]
    for method in hardness_data["method"].unique():
        method_data = hardness_data[hardness_data["method"] == method]
        mean_auc = method_data.groupby("proportion")["auc_roc"].mean()
        std_auc = method_data.groupby("proportion")["auc_roc"].std()
        ax1.errorbar(mean_auc.index, mean_auc.values, yerr=std_auc.values, 
                    label=method, marker='o', capsize=3)
    
    ax1.set_xlabel("Proporção de Perturbação")
    ax1.set_ylabel("AUC-ROC")
    ax1.set_title(f"AUC-ROC por Método - {hardness_type}")
    ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax1.grid(True, alpha=0.3)
    
    ax2 = axes[1]
    for method in hardness_data["method"].unique():
        method_data = hardness_data[hardness_data["method"] == method]
        mean_auc = method_data.groupby("proportion")["auc_prc"].mean()
        std_auc = method_data.groupby("proportion")["auc_prc"].std()
        ax2.errorbar(mean_auc.index, mean_auc.values, yerr=std_auc.values, 
                    label=method, marker='s', capsize=3)
    
    ax2.set_xlabel("Proporção de Perturbação")
    ax2.set_ylabel("AUC-PRC")
    ax2.set_title(f"AUC-PRC por Método - {hardness_type}")
    ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 7. Comparação com Resultados


In [ ]:
comparison_data = []
for hardness_type in HARDNESS_TYPES:
    for prop in PROPORTIONS:
        subset = results_df[
            (results_df["hardness_type"] == hardness_type)
            & (results_df["proportion"] == prop)
        ]
        for method, auc_prc in subset.groupby("method")["auc_prc"].mean().items():
            comparison_data.append({
                "hardness_type": hardness_type,
                "proportion": prop,
                "method": method,
                "mean_auc_prc": auc_prc,
            })

comparison_df = pd.DataFrame(comparison_data)
pivot_table = comparison_df.pivot_table(
    values="mean_auc_prc",
    index=["hardness_type", "method"],
    columns="proportion",
    aggfunc="mean",
)
pivot_table.round(4)
